# Explore the lake with Spark sql and read the query plan

In [1]:
from pyspark.sql import SparkSession

In [2]:
import os
import glob

os.environ["HADOOP_HOME"] = "D:\\hadoop"
os.environ["PATH"] += os.pathsep + "D:\\hadoop\\bin"

In [3]:
spark = (SparkSession.builder.appName("explore")
         .config("spark.ui.showConsoleProgress", False)
         .getOrCreate()
         )
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
spark.read.parquet("../data/lake/silver/order_lines").createOrReplaceTempView("order_lines")
query = """
    SELECT category, COUNT(*) AS lines,
        ROUND(SUM(amount), 2) AS revenue
    FROM order_lines
    WHERE 
        order_date = '2026-11-02'
        AND status = 'paid'
    GROUP BY category
    ORDER BY revenue DESC
"""
result = spark.sql(query)
result.show()
result.explain()

+-----------+-----+----------+
|   category|lines|   revenue|
+-----------+-----+----------+
|     Gaming| 3341|2684490.06|
|    Laptops| 3280|2159758.45|
|Accessories| 3193|1878104.83|
|     Phones| 3270|1819427.45|
|      Audio| 3261|1709616.73|
+-----------+-----+----------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [revenue#41 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(revenue#41 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=208]
      +- HashAggregate(keys=[category#36], functions=[count(1), sum(amount#38)])
         +- Exchange hashpartitioning(category#36, 200), ENSURE_REQUIREMENTS, [plan_id=205]
            +- HashAggregate(keys=[category#36], functions=[partial_count(1), partial_sum(amount#38)])
               +- Project [category#36, amount#38]
                  +- Filter (isnotnull(status#33) AND (status#33 = paid))
                     +- FileScan parquet [status#33,category#36,amount#38,order_date#39] Batched: true, DataFilters

In [ ]:
spark.stop()